# 03: a pretrained ResNet on the artworks (milestone m3)

In m1 our dense network got 0.380 on the test set. In m2 we saw why it stays low: it flattens the image, so it does
not know which pixels are next to each other, and it has 6.3 million weights for only 1 000 training images.

Here we do what Session 7 suggests. We take a ResNet-18 that was already trained on ImageNet (1.2 million photos) and
we adapt it to our 5 genres. We follow Part D of TP07, where the prof does the same thing on CIFAR-10.

## Imports

Same as in notebooks 01 and 02, plus:

- `torchvision.models`: the ResNet-18 and its pretrained weights;
- `time`: to see how long the training takes.

In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import confusion_matrix

DATA_DIR = "data/wikiart_genre5"
np.random.seed(0)
torch.manual_seed(0)
torch.set_num_threads(4)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## Helper functions

`evaluate` is the same function as in TP07 (and in our notebooks 01 and 02): it gives the loss and the accuracy of a
model on a loader. `count_parameters` counts the weights of a model; with `trainable_only=True` it only counts the
ones that the training is allowed to change.

In [2]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total += loss_fn(out, yb).item() * len(yb)
            correct += (out.argmax(dim=1) == yb).sum().item()
            n += len(yb)
    return total / n, correct / n


def count_parameters(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)